# Module 7.3 — Python Internals

### Python Mastery · Track 7: Performance, Internals & C Extensions

Understanding how Python works inside demystifies its behaviour and informs better design. This module covers the object model and the type-object relationship, the attribute lookup chain, and the Global Interpreter Lock, including the experimental free-threaded build that is reshaping Python's concurrency story.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- The object-model and attribute-lookup parts run fully and let you inspect internals directly. The GIL discussion is largely conceptual, since its effects depend on the interpreter build; the runnable cells illustrate what can be observed in this standard build.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Explain the type-object relationship and the role of `type` and `object`.
2. Trace the attribute lookup chain: instance, class, base classes, metaclass.
3. Explain how data and non-data descriptors affect lookup priority.
4. Describe what the Global Interpreter Lock protects and its consequences.
5. Explain free-threaded CPython (PEP 703) and what it changes.

**Prerequisites:** Tracks 1 to 6 (especially Modules 3.4 and 4.4), and Modules 7.1 to 7.2.

---

## Part 1 · The Object Model: Everything Is an Object

In Python, every value is an object with a type, and types are themselves objects. Two special objects sit at the root: `object`, the base class from which all classes ultimately inherit, and `type`, the metaclass of which all classes (including `object`) are instances. This circular-looking but consistent arrangement is the foundation of the whole model.

In [ ]:
# Every value has a type; types are objects too.
print("type of 42:", type(42))
print("type of int:", type(int))            # int is an instance of type
print("type of type:", type(type))          # type is an instance of itself

# object is the ultimate base class; type itself inherits from object.
print("int's bases:", int.__bases__)         # (object,)
print("object's bases:", object.__bases__)   # () -- the root
print("type inherits from object:", issubclass(type, object))
print("object is an instance of type:", isinstance(object, type))

Each object carries identity (`id`), a type, and a value. The `is` operator compares identity, while `==` compares value. Small integers and short strings are often cached (interned) by CPython, so identical literals may share one object, an implementation detail you should never rely on for correctness.

In [ ]:
# Identity versus equality.
a = [1, 2, 3]
b = [1, 2, 3]
print("equal values:", a == b)              # True: same contents
print("same object:", a is b)               # False: distinct objects

# CPython caches small integers (an implementation detail).
x = 256
y = 256
print("small int cached:", x is y)           # typically True
big = 10**6
print("large int not cached:", (10**6) is big)   # typically False

## Part 2 · The Attribute Lookup Chain

When you access `obj.name`, Python searches in a defined order. Simplified: it checks the type for a **data descriptor** first, then the instance's own `__dict__`, then the class and its base classes (following the method resolution order), and finally falls back to `__getattr__` if defined. Understanding this order explains why methods, class attributes, and instance attributes interact as they do.

In [ ]:
class Base:
    shared = "from Base"             # class attribute on a parent

class Child(Base):
    role = "from Child"              # class attribute on the class itself

c = Child()
c.own = "from instance"              # instance attribute

# Resolution order in action:
print("instance attribute:", c.own)          # found in the instance __dict__
print("class attribute:", c.role)            # found on Child
print("inherited attribute:", c.shared)      # found on Base via the MRO

# The method resolution order Python follows for lookup:
print("MRO:", [cls.__name__ for cls in Child.__mro__])

An instance attribute normally **shadows** a class attribute of the same name, because the instance `__dict__` is checked before the class. This is the common case and explains how per-instance state overrides class defaults.

In [ ]:
class Config:
    timeout = 30                     # a class-level default

a = Config()
b = Config()
b.timeout = 5                        # set an instance attribute on b only

print("a uses the class default:", a.timeout)    # 30
print("b shadows it per-instance:", b.timeout)   # 5
print("the class attribute is unchanged:", Config.timeout)   # 30

## Part 3 · Descriptors and Lookup Priority

The one exception to "instance shadows class" is a **data descriptor**: an object on the class defining both `__get__` and `__set__` (such as a `property`). Data descriptors take priority over the instance dictionary, which is why a `property` always runs its getter even if an instance somehow has an attribute of the same name. A **non-data descriptor** (only `__get__`, such as a plain method) has lower priority than the instance dictionary. This priority rule, introduced functionally in Module 3.4, is the heart of how attribute access works.

In [ ]:
class DataDescriptor:
    """Defines __get__ AND __set__, so it is a data descriptor (high priority)."""
    def __get__(self, instance, owner):
        return "value from the data descriptor"
    def __set__(self, instance, value):
        instance.__dict__["managed"] = value

class Example:
    managed = DataDescriptor()

e = Example()
e.managed = "tried to set on the instance"
# Even with an instance __dict__ entry, the data descriptor's __get__ wins.
print("lookup returns:", e.managed)
print("instance dict still holds:", e.__dict__.get("managed"))

In [ ]:
# A non-data descriptor (only __get__) has LOWER priority than the instance dict.
class NonData:
    def __get__(self, instance, owner):
        return "value from the non-data descriptor"

class Demo:
    attr = NonData()

d = Demo()
print("before instance assignment:", d.attr)       # uses the descriptor
d.__dict__["attr"] = "instance value"               # now the instance dict has it
print("after instance assignment:", d.attr)         # the instance value wins

## Part 4 · The Global Interpreter Lock

CPython's **Global Interpreter Lock** (GIL) is a single lock that allows only one thread to execute Python bytecode at a time. It exists because CPython's memory management (especially reference counting) is not thread-safe without it; the GIL makes the interpreter simpler and single-threaded code faster, at the cost of true multi-core parallelism for pure-Python work.

The practical consequences, seen in Track 5: threads do not speed up CPU-bound Python code (only one runs bytecode at a time), but they do help input/output-bound code, because the GIL is released while a thread waits. For CPU parallelism you use multiple processes, or drop into C, where the GIL can be released around heavy computation.

In [ ]:
import sys

# We can confirm we are on a standard, GIL-enabled CPython build.
print("implementation:", sys.implementation.name)
print("version:", sys.version.split()[0])

# Newer builds expose an API to query the GIL; older ones do not have it.
if hasattr(sys, "_is_gil_enabled"):
    print("GIL currently enabled:", sys._is_gil_enabled())
else:
    print("this build has no GIL-toggle API, so the GIL is always enabled here")

A key point for Module 7.5: a C extension can **release the GIL** around long computations or blocking calls (using the `Py_BEGIN_ALLOW_THREADS` / `Py_END_ALLOW_THREADS` macros), letting other Python threads run meanwhile. This is exactly how libraries such as NumPy achieve parallelism for heavy numeric work despite the GIL: the heavy loop runs in C with the lock released.

## Part 5 · Free-Threaded CPython (PEP 703)

The GIL is no longer a permanent fixture. **PEP 703** introduces an experimental **free-threaded** build of CPython (available from version 3.13 onward) that removes the GIL entirely, allowing Python threads to run in genuine parallel across cores. This is a major, multi-year transition.

What it means going forward:

- In a free-threaded build, CPU-bound multi-threaded Python code can finally use multiple cores, narrowing the gap with multiprocessing for shared-memory workloads.
- It is **opt-in and experimental**: it is a separate build (often labelled with a `t` suffix, such as `python3.13t`), and single-threaded code may run somewhat slower because reference counting must become thread-safe.
- Code that relied on the GIL for implicit atomicity may need explicit locks, so libraries and applications are adapting gradually.
- The standard distribution remains GIL-enabled for now; the two builds coexist while the ecosystem catches up.

The takeaway is directional: the concurrency advice in Track 5 (threads for input/output, processes for CPU) holds for today's default interpreter, but the free-threaded build signals that true multi-core threading is becoming part of Python's future. The cell below reports what the current build supports.

In [ ]:
import sys

# Report whether this interpreter is a free-threaded build.
# The 'gil' attribute on sys.flags exists on builds aware of the option.
is_freethreaded = getattr(sys.flags, "gil", None) == 0
print("running a free-threaded build:", is_freethreaded)
print("interpreter:", sys.implementation.name, sys.version.split()[0])

if not is_freethreaded:
    print("this is the standard GIL-enabled build; PEP 703 builds are a separate,")
    print("experimental download (for example 'python3.13t') that removes the GIL")

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Type relationships (Easy)

In [ ]:
for value in [42, "hi", [1], {}, int]:
    print(f"{str(value):>6} -> type {type(value).__name__}")
print("all are instances of object:", all(isinstance(v, object) for v in [42, "hi", [1]]))

### Example 2 — Identity versus equality (Easy)

In [ ]:
a = [1, 2]
b = a
c = [1, 2]
print("a is b:", a is b)       # same object
print("a is c:", a is c)       # different objects
print("a == c:", a == c)       # equal values

### Example 3 — Tracing attribute resolution (Medium)

In [ ]:
class A:
    x = "A.x"
class B(A):
    y = "B.y"

obj = B()
obj.z = "instance.z"
for name in ["z", "y", "x"]:
    print(f"{name}: {getattr(obj, name)}")
print("MRO:", [c.__name__ for c in B.__mro__])

### Example 4 — Instance shadowing a class attribute (Medium)

In [ ]:
class Counter:
    start = 0

c1 = Counter()
c2 = Counter()
c2.start = 100               # instance attribute on c2
print("c1.start:", c1.start)        # class default
print("c2.start:", c2.start)        # shadowed
print("Counter.start:", Counter.start)

### Example 5 — A property as a data descriptor (Difficult)

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def fahrenheit(self):               # property = data descriptor, high priority
        return self._celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self._celsius = (value - 32) * 5 / 9

t = Temperature(25)
print("25C in F:", t.fahrenheit)
t.fahrenheit = 212                       # uses the setter
print("after setting 212F, celsius is:", t._celsius)
# The property's __get__/__set__ always run; an instance cannot shadow it.

### Example 6 — Observing GIL behaviour with threads (Difficult)

In [ ]:
import threading, time

# CPU-bound work: the GIL means threads do NOT speed this up on a standard build.
def cpu_work(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

n = 2_000_000

start = time.perf_counter()
cpu_work(n); cpu_work(n)                  # sequential
sequential = time.perf_counter() - start

start = time.perf_counter()
threads = [threading.Thread(target=cpu_work, args=(n,)) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()                # "concurrent" threads
threaded = time.perf_counter() - start

print(f"sequential: {sequential:.3f}s")
print(f"threaded:   {threaded:.3f}s")
print("on a GIL build these are similar: threads cannot run Python bytecode in parallel")

---

## Exercises

**Exercise 1 (Easy).** Print the type of `3.14`, the type of `float`, and confirm with `isinstance` that `float` is an instance of `type`.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Create two equal but distinct lists and show, using `is` and `==`, that they are equal in value but not identical.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Define a base class with attribute `kind = "base"` and a subclass; create an instance, give it an instance attribute `name`, and print the instance attribute, the inherited attribute, and the class MRO.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Show instance shadowing: a class with attribute `rate = 1.0`, two instances, one of which sets `rate` to `2.0`, and print both instances' `rate` plus the unchanged class attribute.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a class with a read-only `area` property computed from a stored `radius`, demonstrate it, and explain in a comment why the instance cannot override the property by assignment (you may attempt the assignment and catch the error).

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
print(type(3.14))
print(type(float))
print(isinstance(float, type))

**Solution 2**

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
print("a == b:", a == b)
print("a is b:", a is b)

**Solution 3**

In [ ]:
class Base:
    kind = "base"
class Sub(Base):
    pass

s = Sub()
s.name = "example"
print("instance:", s.name)
print("inherited:", s.kind)
print("MRO:", [c.__name__ for c in Sub.__mro__])

**Solution 4**

In [ ]:
class Pricing:
    rate = 1.0

a = Pricing()
b = Pricing()
b.rate = 2.0
print("a.rate:", a.rate)
print("b.rate:", b.rate)
print("Pricing.rate:", Pricing.rate)

**Solution 5**

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius
    @property
    def area(self):
        return 3.14159 * self.radius ** 2

c = Circle(2)
print("area:", round(c.area, 2))
# A property with only a getter is a data descriptor (it also defines __set__,
# which raises), so assigning to it is blocked rather than shadowed.
try:
    c.area = 100
except AttributeError as e:
    print("cannot assign:", e)

---

## Summary and Key Points

- Everything is an object with identity, type, and value; `object` is the root base class and `type` is the metaclass of all classes (and an instance of itself). `is` compares identity, `==` compares value; small-integer and string caching is an implementation detail.
- Attribute lookup follows a defined order: data descriptors on the type, then the instance `__dict__`, then the class and its bases via the MRO, then `__getattr__`; instance attributes normally shadow class attributes.
- Data descriptors (`__get__` and `__set__`, such as `property`) outrank the instance dictionary, so they always run; non-data descriptors (only `__get__`, such as methods) rank below it.
- The Global Interpreter Lock lets only one thread run Python bytecode at a time, protecting CPython's memory management; threads help input/output-bound code but not CPU-bound Python code, and C extensions can release the GIL around heavy work.
- PEP 703 introduces an experimental, opt-in free-threaded build (3.13+) that removes the GIL for true multi-core threading; the standard build remains GIL-enabled while the ecosystem adapts, but it signals Python's concurrency direction.

### Next module: 7.4 — Bytecode & AST Analysis

The next module looks at how Python compiles source to bytecode, how to inspect it with `dis`, and how to analyse and transform code through the abstract syntax tree.